In [1]:
import datasets
# cache_path = '/data/kcl/lpy/hf'
data_path = "math-dataset/DeepScaleR-Preview-Dataset"
dataset = datasets.load_dataset(data_path,
                                # cache_dir=cache_path
                               )

README.md:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

deepscaler.json:   0%|          | 0.00/21.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40315 [00:00<?, ? examples/s]

In [8]:
dataset['train'][0]

{'problem': 'The operation $\\otimes$ is defined for all nonzero numbers by $a \\otimes b = \\frac{a^{2}}{b}$. Determine $[(1 \\otimes 2) \\otimes 3] - [1 \\otimes (2 \\otimes 3)]$.',
 'answer': '-\\frac{2}{3}',
 'solution': '1. **Apply the operation $\\otimes$ to the innermost parentheses first:**\n   \\[\n   (1 \\otimes 2) \\otimes 3 = \\left(\\frac{1^2}{2}\\right) \\otimes 3 = \\frac{1}{2} \\otimes 3\n   \\]\n   \\[\n   1 \\otimes (2 \\otimes 3) = 1 \\otimes \\left(\\frac{2^2}{3}\\right) = 1 \\otimes \\frac{4}{3}\n   \\]\n\n2. **Calculate each part using the definition of $\\otimes$:**\n   \\[\n   \\frac{1}{2} \\otimes 3 = \\frac{\\left(\\frac{1}{2}\\right)^2}{3} = \\frac{\\frac{1}{4}}{3} = \\frac{1}{12}\n   \\]\n   \\[\n   1 \\otimes \\frac{4}{3} = \\frac{1^2}{\\frac{4}{3}} = \\frac{1}{\\frac{4}{3}} = \\frac{3}{4}\n   \\]\n\n3. **Subtract the two results:**\n   \\[\n   \\left(\\frac{1}{12}\\right) - \\left(\\frac{3}{4}\\right) = \\frac{1}{12} - \\frac{9}{12} = -\\frac{8}{12} = -\\f

In [5]:
system_prompt = '''Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\n'''

In [12]:
    train_dataset = dataset["train"]
    def make_map_fn(split):
        def process_fn(example, idx):
            extra_info = {'index':idx}
            extra_info["need_tools_kwargs"] = True
            extra_info["tools_kwargs"] = {
                "code_interpreter": {
                    "create_kwargs": {
                        "ground_truth": example["answer"],
                    },
                },
            }
            extra_info['split'] = None
            extra_info['raw_problem'] = example['problem']
            example["extra_info"] = extra_info
            example['data_source'] = 'aime_deepscale_r'
            example['ability'] = 'MATH'
            example['reward_model'] = {
                'ground_truth': example["answer"],
                'style': 'rule-lighteval/MATH_v2'
            }
            example['prompt'] = [{
                'content' : f'{system_prompt}{example['problem']}\n\nRemember to put your answer on its own line after "Answer:".',
                'role' : 'user'
            }]
            return example
        return process_fn

    train_dataset = train_dataset.map(function=make_map_fn("train"), with_indices=True)

In [13]:
train_dataset

Dataset({
    features: ['problem', 'answer', 'solution', 'extra_info', 'data_source', 'ability', 'reward_model', 'prompt'],
    num_rows: 40315
})

In [15]:
    import os
    save_dir = '/home/v-peiyangliu/msranlphot/liupeiyang/dataset/deepscale_r'
    train_dataset.to_parquet(os.path.join(save_dir, "train_verl.parquet"))

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

53258775

In [18]:
__index_level_0__list = []
for item in train_dataset:
    __index_level_0__list.append(item['__index_level_0__'])

In [23]:
len(set(__index_level_0__list))

960

In [21]:
len(train_dataset)

960

In [13]:
datasets.__version__

'4.5.0'